# 05 — Fine-tuned DistilBERT (Approach 5 of 5)

A pretrained Transformer, fine-tuned end-to-end. This is the only approach using
**subword tokenization**, which directly addresses two EDA findings that word-level
models (notebooks 03-04) cannot: finding #9 (39.7% of test-set content words are
out-of-vocabulary relative to train) and finding #8 (56.4% of the vocabulary are
singleton words). Subwords degrade gracefully on rare/misspelled/unseen words instead of
mapping them all to a single `<unk>` token. Self-attention across the whole sequence is
also well suited to finding #10 (all classes share the same core vocabulary — the model
must use context, not just word presence).

We use `distilbert-base-uncased` (40% fewer parameters than BERT-base, ~60% faster)
because training is **CPU-only** in this environment — full BERT-base fine-tuning would
be impractically slow. `max_length=32` follows EDA finding #4 (median tweet length is
17 words; 32 sub-word tokens comfortably covers the vast majority of tweets after
WordPiece tokenization while keeping the quadratic-in-length attention cost down).

**Compute-budget note:** a CPU throughput benchmark (see `bench_distilbert.py` in the
project's build tooling) measured roughly 150-240ms/sample for a forward+backward pass
at this sequence length, i.e. ~25-30 minutes per epoch over the full 8,205-row training
set on this 4-core CPU. We therefore fine-tune for **1 epoch** rather than the 2-3 that
would be standard on a GPU — this is a real, disclosed limitation (see Section 6 of the
report), not a modelling choice we'd make given more compute.


In [1]:
import sys
sys.path.append('../src')
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

from preprocessing import LABEL_TO_IDX, CLASS_NAMES
from train_utils import compute_class_weights
from eval_utils import evaluate, set_seed

set_seed(42)
torch.set_num_threads(4)
device = 'cpu'

train_df = pd.read_csv('../data/processed/train_split.csv')
val_df = pd.read_csv('../data/processed/val_split.csv')
train_labels = train_df['label'].map(LABEL_TO_IDX).tolist()
val_labels = val_df['label'].map(LABEL_TO_IDX).tolist()
print(len(train_df), len(val_df))


8205 1449


## Tokenize with the pretrained DistilBERT subword tokenizer

In [2]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 32

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        list(texts), padding='max_length', truncation=True, max_length=MAX_LEN,
        return_tensors='pt',
    )

train_enc = tokenize(train_df['clean_text'])
val_enc = tokenize(val_df['clean_text'])
print(train_enc['input_ids'].shape)


torch.Size([8205, 32])


In [3]:
class EncodedDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = EncodedDataset(train_enc, train_labels)
val_ds = EncodedDataset(val_enc, val_labels)


## Fine-tune
CPU-only budget: **1 epoch**, batch size 32, AdamW with a low learning rate (2e-5,
standard for Transformer fine-tuning). Class-weighted loss for the same imbalance
reason as the other neural models (EDA finding #1). Progress and a JSON checkpoint are
written every 50 steps to `reports/results/distilbert_train_progress.json` so partial
progress survives even if this cell is interrupted.


In [ ]:
import json as _json
from pathlib import Path as _Path

model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)

class_weights = compute_class_weights(train_labels).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

BATCH_SIZE = 32
EPOCHS = 1
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

PROGRESS_PATH = _Path('../reports/results/distilbert_train_progress.json')
PROGRESS_PATH.parent.mkdir(parents=True, exist_ok=True)

history = []
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    total_loss = 0.0
    n_seen = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        labels = batch.pop('labels').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        n_seen += labels.size(0)
        if step % 50 == 0:
            elapsed = time.time() - t0
            print(f'  epoch {epoch} step {step}/{len(train_loader)} loss={loss.item():.4f} '
                  f'running_avg_loss={total_loss/n_seen:.4f} elapsed={elapsed:.0f}s')
            with open(PROGRESS_PATH, 'w') as f:
                _json.dump({'epoch': epoch, 'step': step, 'total_steps': len(train_loader),
                            'running_avg_loss': total_loss / n_seen, 'elapsed_s': elapsed}, f)
    train_loss = total_loss / len(train_ds)

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch.pop('labels').to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            val_preds.exte
            nd(outputs.logits.argmax(dim=1).cpu().tolist())
            val_true.extend(labels.cpu().tolist())
    from sklearn.metrics import f1_score
    val_f1 = f1_score(val_true, val_preds, average='macro')
    elapsed = time.time() - t0
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_macro_f1': val_f1})
    print(f'Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} val_macro_f1={val_f1:.4f} ({elapsed:.0f}s)')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1 step 0/257 loss=1.1291 running_avg_loss=1.1291 elapsed=20s


  epoch 1 step 50/257 loss=0.9568 running_avg_loss=1.0393 elapsed=380s


  epoch 1 step 100/257 loss=0.7084 running_avg_loss=0.9614 elapsed=654s


  epoch 1 step 150/257 loss=0.7340 running_avg_loss=0.9175 elapsed=893s


  epoch 1 step 200/257 loss=0.8583 running_avg_loss=0.8831 elapsed=1132s


  epoch 1 step 250/257 loss=0.7367 running_avg_loss=0.8660 elapsed=1371s


Epoch 1/1 | train_loss=0.8648 val_macro_f1=0.5912 (1470s)


## Evaluate on validation set

In [5]:
result = evaluate(val_true, val_preds, 'DistilBERT_FineTuned')



=== DistilBERT_FineTuned ===
Macro-F1: 0.5912  Weighted-F1: 0.6732  Accuracy: 0.6715
              precision    recall  f1-score   support

    Negative       0.29      0.55      0.38       154
     Neutral       0.76      0.88      0.81       704
    Positive       0.80      0.46      0.58       591

    accuracy                           0.67      1449
   macro avg       0.61      0.63      0.59      1449
weighted avg       0.72      0.67      0.67      1449



## Summary
Results saved to `reports/results/DistilBERT_FineTuned.json`; confusion matrix to
`figures/cm_distilbert_finetuned.png`. Model weights are **not** committed to the repo
(too large for git) — re-run this notebook to reproduce them; `transformers`
`set_seed`/our `set_seed(42)` keeps results close to reproducible.
